# Lab 13 — Discipline-Specific Agents

## Objective

Use vibration and temperature trends to classify machine conditions and recommend human action.

## Introduction and concept

diagnose combines recent values with prior-average trends and provides an operator recommendation for alarms.

## Real-world use case

A maintenance technician needs to triage abnormal readings from a rotating machine.

## Technology stack and dependencies

Python statistics and dataclasses. Use Python 3.11 or newer and a Jupyter kernel from the environment where you installed the [assignment requirements](../requirements.txt). Start the notebook from its lab directory.

## Architecture and workflow

Validate readings → Compare against baseline → Assign condition → Recommend review


## Step 1 — Implement the components

diagnose combines recent values with prior-average trends and provides an operator recommendation for alarms.

In [1]:
from dataclasses import dataclass
from math import isfinite
from statistics import mean

@dataclass(frozen=True)
class Reading:
    vibration_mm_s: float
    temperature_c: float

def diagnose(readings):
    if len(readings) < 3 or any(
        not isfinite(r.vibration_mm_s) or r.vibration_mm_s < 0 or not isfinite(r.temperature_c)
        for r in readings
    ):
        raise ValueError("Provide at least three valid sensor readings")
    recent = readings[-1]
    baseline = readings[:-1]
    vibration_change = recent.vibration_mm_s - mean(r.vibration_mm_s for r in baseline)
    heat_change = recent.temperature_c - mean(r.temperature_c for r in baseline)
    if recent.vibration_mm_s >= 8 or recent.temperature_c >= 90:
        level, action = "alarm", "Request operator review of the shutdown procedure"
    elif recent.vibration_mm_s >= 5 or recent.temperature_c >= 75 or vibration_change >= 2 or heat_change >= 8:
        level, action = "warning", "Schedule inspection and verify the sensor readings"
    else:
        level, action = "normal", "Continue monitoring"
    return {"condition": level, "vibration_change": round(vibration_change, 1),
            "temperature_change": round(heat_change, 1), "recommendation": action,
            "automatic_shutdown": False}

## Step 2 — Run sample inputs and inspect actual output

Compare normal, warning and alarm fixture readings. Verify invalid input handling and that no case automatically shuts down equipment.

In [2]:
def main():
    baseline = [Reading(2.5, 62), Reading(3, 64)]
    for last, expected in [
        (Reading(3, 65), "normal"), (Reading(4.8, 72), "warning"), (Reading(8.2, 87), "alarm"),
    ]:
        result = diagnose([*baseline, last])
        print(result)
        assert result["condition"] == expected and result["automatic_shutdown"] is False
    try:
        diagnose([Reading(float("nan"), 50)] * 3)
    except ValueError:
        print("PASS: invalid sensor readings and human-reviewed alarm")
    else:
        raise AssertionError("Expected invalid sensor data to be rejected")

main()

{'condition': 'normal', 'vibration_change': 0.2, 'temperature_change': 2, 'recommendation': 'Continue monitoring', 'automatic_shutdown': False}
{'condition': 'warning', 'vibration_change': 2.0, 'temperature_change': 9, 'recommendation': 'Schedule inspection and verify the sensor readings', 'automatic_shutdown': False}
{'condition': 'alarm', 'vibration_change': 5.4, 'temperature_change': 24, 'recommendation': 'Request operator review of the shutdown procedure', 'automatic_shutdown': False}
PASS: invalid sensor readings and human-reviewed alarm


## Testing and validation

Compare normal, warning and alarm fixture readings. Verify invalid input handling and that no case automatically shuts down equipment. Assertions in the preceding cell check the demonstrated behavior and stop execution on a failed expectation.

## Limitations

Thresholds are fictional training values. The program never performs a shutdown or controls equipment.

## How this relates to ExamPilot

Domain analogue. ExamPilot applies course-specific assessment and SQL result comparison in backend/app/agents/evaluation_agent/sql_grader.py. Predictive maintenance is the separate disciplinary example.

## Conclusion

Use vibration and temperature trends to classify machine conditions and recommend human action. Use the trace and assertions to explain which decisions the workflow makes.